# 12 因果推論 — 參考解答

松柏護理之家退伍軍人症群聚事件因果推論練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "dead").astype(int)

## 題目 1：水療暴露的歸因風險

In [ ]:
# 水療暴露
hydro_exp = df[df["hydrotherapy_use"] == 1]
hydro_unexp = df[df["hydrotherapy_use"] == 0]

risk_hydro_exp = hydro_exp["infected"].mean()
risk_hydro_unexp = hydro_unexp["infected"].mean()
risk_total = df["infected"].mean()

AR_hydro = risk_hydro_exp - risk_hydro_unexp
PAR_hydro = risk_total - risk_hydro_unexp
PAR_pct_hydro = PAR_hydro / risk_total * 100

print("=== 水療暴露 ===")
print(f"使用者侵襲率：{risk_hydro_exp:.1%}")
print(f"非使用者侵襲率：{risk_hydro_unexp:.1%}")
print(f"AR = {AR_hydro:.3f}")
print(f"PAR% = {PAR_pct_hydro:.1f}%")

# 淋浴暴露（比較用）
shower_exp = df[df["shower_use"] == 1]
shower_unexp = df[df["shower_use"] == 0]
risk_sh_exp = shower_exp["infected"].mean()
risk_sh_unexp = shower_unexp["infected"].mean()
AR_shower = risk_sh_exp - risk_sh_unexp
PAR_shower = risk_total - risk_sh_unexp
PAR_pct_shower = PAR_shower / risk_total * 100

print(f"\n=== 淋浴暴露（比較）===")
print(f"AR = {AR_shower:.3f}")
print(f"PAR% = {PAR_pct_shower:.1f}%")

print(f"\n=== 比較 ===")
if abs(AR_shower) > abs(AR_hydro):
    print("\u2192 淋浴暴露的 AR 較大，對感染的貢獻更高")
else:
    print("\u2192 水療暴露的 AR 較大")

print("\n\u2192 AR 代表『如果因果關係成立，消除暴露可減少的風險量』")
print("\u2192 前提：(1) 因果關係成立 (2) 無交絡因子 (3) 暴露是可消除的")

## 題目 2：改變 DiD 介入日期

In [ ]:
cases = df[df["infected"] == 1].copy()
all_dates = pd.date_range("2026-01-12", "2026-01-28", freq="D")

# 介入組 / 對照組
treated_mask = (cases["floor"].isin([2, 3])) & (cases["wing"] == "B")
treated_daily = cases[treated_mask].groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)
control_daily = cases[~treated_mask].groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)

# 比較不同介入日
for cutoff in ["2026-01-22", "2026-01-25"]:
    panel = pd.DataFrame({
        "date": list(all_dates) * 2,
        "treated": [1] * len(all_dates) + [0] * len(all_dates),
        "daily_cases": list(treated_daily.values) + list(control_daily.values),
    })
    panel["post"] = (panel["date"] >= cutoff).astype(int)

    model = smf.ols("daily_cases ~ treated + post + treated:post", data=panel).fit()
    coef = model.params["treated:post"]
    pval = model.pvalues["treated:post"]

    print(f"介入日 = {cutoff}: treated:post = {coef:.3f}, p = {pval:.4f}")

print("\n\u2192 改變介入日期會影響 DiD 結果")
print("\u2192 原因：介入前後的觀察天數不同，前後病例數分布也不同")
print("\u2192 選擇介入日必須基於實際事件（真的消毒了），不能隨意挑選")
print("\u2192 如果亂挑介入日做出顯著結果 = p-hacking")

## 題目 3（挑戰題）：碰撞因子偏誤的實證

In [ ]:
from epi_learning import risk_ratio

# 全體 RR
ct_all = pd.crosstab(df["shower_use"], df["infected"])
rr_all = risk_ratio(ct_all)
print(f"=== 全體 RR (shower \u2192 infected) ===")
print(f"RR = {rr_all:.3f}")

# 只取住院者
hosp = df[df["hospitalized"] == 1].copy()
print(f"\n住院者：{len(hosp)} 人")
print(f"住院者中 shower_use 分布：{hosp['shower_use'].value_counts().to_dict()}")
print(f"住院者中 infected 分布：{hosp['infected'].value_counts().to_dict()}")

# 住院者全部都是感染者嗎？
if hosp["infected"].nunique() == 1:
    print("\n\u2192 住院者全部都是感染者（infected=1），無法計算 RR")
    print("\u2192 這正是碰撞因子偏誤的極端情況！")
    print("\u2192 因為只有感染且嚴重的人才會住院")
    print("\u2192 在住院者中，shower_use 和 infected 的關係被扭曲")
else:
    ct_hosp = pd.crosstab(hosp["shower_use"], hosp["infected"])
    rr_hosp = risk_ratio(ct_hosp)
    print(f"住院者 RR = {rr_hosp:.3f}")
    print(f"全體 RR = {rr_all:.3f}")
    print(f"\n\u2192 限定住院者後 RR 改變了！")
    print("\u2192 這就是碰撞因子偏誤（collider bias）")

print("\n=== 碰撞因子偏誤解釋 ===")
print("hospitalized \u2190 severity \u2190 infection")
print("hospitalized \u2190 infection")
print("\u2192 hospitalized 是碰撞因子，受 severity 和 infection 共同影響")
print("\u2192 條件化碰撞因子（只看住院者）= 打開一條假性路徑")
print("\u2192 結果：在住院者中，shower_use 和 infection 的關係被扭曲")

### 解讀

- **AR/PAR**：水療 vs 淋浴的歸因風險不同，反映不同暴露途徑的貢獻。淋浴是產生退伍軍人菌氣溶膠的主要途徑
- **DiD 介入日**：結果對介入日期敏感。正確的做法是用實際介入日期，不是事後挑選最顯著的
- **碰撞因子**：只分析住院者 = 對碰撞因子做條件化，會產生 selection bias。這是觀察性研究中常見的陷阱
- **因果推論的限制**：在觀察性資料中，我們永遠無法完全確定因果關係。DAG 和統計方法只能幫我們辨識和減少偏誤，但不能消除所有未觀測到的交絡因子